# Normalized Discounted Cumulative Gain at K (NDCG@K) — Mathematical & Conceptual Guide

## 1. Overview & Core Concept

While **MRR (Mean Reciprocal Rank)** stops after the *first* hit, and **MAP@K** assumes items are strictly *binary* (either 100% relevant or 0% relevant), **NDCG@K** is the **reigning industry gold standard** across commercial recommendation engines and search platforms (Netflix, Spotify, Google, Amazon).

NDCG is built around two fundamental real-world truths:
1. **Relevance is Graded, Not Binary:** A 5-star movie is vastly more valuable to recommend than a mediocre 3-star movie, and an item purchased is far more significant than an item just clicked.
2. **Position Matters Non-Linearly:** Users examine the top items with intense focus, but rarely scroll to the bottom. High-value items placed near the top must be heavily rewarded, while placing them near the bottom must be severely penalized.

## 2. The 4 Fundamental Building Blocks of NDCG

To understand NDCG, we build it mathematically through four progressive layers:

---

### Layer 1: Gain (G)
The raw numerical relevance score of an item at rank position $i$. 
In modern production systems, we use the **exponential gain** formulation:
$$\text{Gain}_i = 2^{\text{rel}_i} - 1$$
*(Where $\text{rel}_i$ is the graded rating, e.g., $0$ to $5$ stars, or interaction weight).*

---

### Layer 2: Cumulative Gain at K (CG@K)
The simple sum of gains across the top $K$ items, **ignoring their rank position**:
$$\text{CG@K} = \sum_{i=1}^K (2^{\text{rel}_i} - 1)$$
* **The Fatal Flaw of CG:** It is completely order-blind! An item at Rank 1 provides the exact same score as an item at Rank $K$.

---

### Layer 3: Discounted Cumulative Gain at K (DCG@K)
Introduces a **logarithmic discount** that divides each item's gain by a value that grows with its rank position $i$:
$$\text{DCG@K} = \sum_{i=1}^K \frac{2^{\text{rel}_i} - 1}{\log_2(i + 1)}$$

* At Rank 1 ($i=1$): Denominator is $\log_2(1 + 1) = \log_2(2) = 1.0$ $\rightarrow$ **Full 100% value**.
* At Rank 3 ($i=3$): Denominator is $\log_2(3 + 1) = \log_2(4) = 2.0$ $\rightarrow$ **Value cut in half!**

---

### Layer 4: Normalized Discounted Cumulative Gain at K (NDCG@K)
DCG produces an unnormalized raw score (e.g., $23.79$) that varies wildly depending on the query or user rating distribution. To normalize it into a standardized score between **$0.0$ and $1.0$**, we divide DCG by the **Ideal DCG (IDCG@K)**:

$$\text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}$$

* **$\text{IDCG@K}$** is the DCG score achieved if the items were placed in **perfect descending order of relevance**.

## 3. The Math Behind the Formulas

### Why the Logarithmic Discount $\frac{1}{\log_2(i + 1)}$?
The discount factor $\frac{1}{\log_2(i + 1)}$ mirrors **human eye-tracking behavior**:

| Rank ($i$) | Logarithm Base 2 | Discount Multiplier $\frac{1}{\log_2(i+1)}$ | Penalty / Value Retained |
| :---: | :---: | :---: | :---: |
| **Rank 1** | $\log_2(2) = 1.000$ | $\frac{1}{1.000} = \mathbf{1.000}$ | **100% full credit** |
| **Rank 2** | $\log_2(3) \approx 1.585$ | $\frac{1}{1.585} \approx \mathbf{0.631}$ | **Lost 36.9% of value** |
| **Rank 3** | $\log_2(4) = 2.000$ | $\frac{1}{2.000} = \mathbf{0.500}$ | **Value cut in half (50%)** |
| **Rank 7** | $\log_2(8) = 3.000$ | $\frac{1}{3.000} \approx \mathbf{0.333}$ | **Value cut to one-third** |
| **Rank 15** | $\log_2(16) = 4.000$ | $\frac{1}{4.000} = \mathbf{0.250}$ | **Value cut to one-fourth** |

* Notice that dropping from **Rank 1 to Rank 2** loses nearly **$37\%$** of credit.
* Dropping from **Rank 7 to Rank 8** barely changes the score.
* This matches reality: users scrutinize the top 3 items, but care very little about the difference between position 8 and position 9.

---

### Why Exponential Gain ($2^{\text{rel}} - 1$) instead of Linear Gain ($\text{rel}$)?
* If $\text{rel} = 3$ (Good item): $\text{Gain} = 2^3 - 1 = \mathbf{7}$
* If $\text{rel} = 5$ (Masterpiece item): $\text{Gain} = 2^5 - 1 = \mathbf{31}$
* The 5-star item is rewarded with **over $4.4\times$ more points** than the 3-star item! This forces algorithms to prioritize true customer favorites rather than settling for mediocre "safe" recommendations.

## 4. Concrete Mathematical Example (Step-by-Step)

Suppose a movie streaming recommendation engine suggests **$K = 4$ movies** to a user.

### Graded Relevance Ground Truth Scale:
* **5** = Masterpiece / Loved it
* **3** = Good / Enjoyed it
* **2** = Mediocre / Average
* **0** = Disliked / Irrelevant

### The Model's Actual Ranked Recommendations:
* **Rank 1:** Movie A (Actual $\text{rel} = 3$)
* **Rank 2:** Movie B (Actual $\text{rel} = 0$)
* **Rank 3:** Movie C (Actual $\text{rel} = 5$) *(The 5-star masterpiece was buried at Rank 3!)*
* **Rank 4:** Movie D (Actual $\text{rel} = 2$)

$$\mathbf{rel}_{\text{model}} = [3, 0, 5, 2]$$

### Step 1: Calculate the Model's DCG@4

For each rank position $i = 1, 2, 3, 4$, compute $\frac{2^{\text{rel}_i} - 1}{\log_2(i + 1)}$:

| Rank ($i$) | Recommended Item | Actual Relevance ($\text{rel}_i$) | Numerator ($2^{\text{rel}} - 1$) | Denominator ($\log_2(i + 1)$) | Discounted Gain Added |
| :---: | :---: | :---: | :---: | :---: | :---: |
| **1** | Movie A | **3** | $2^3 - 1 = \mathbf{7}$ | $\log_2(2) = \mathbf{1.000}$ | $\frac{7}{1.000} = \mathbf{7.000}$ |
| **2** | Movie B | **0** | $2^0 - 1 = \mathbf{0}$ | $\log_2(3) \approx \mathbf{1.585}$ | $\frac{0}{1.585} = \mathbf{0.000}$ |
| **3** | Movie C | **5** | $2^5 - 1 = \mathbf{31}$ | $\log_2(4) = \mathbf{2.000}$ | $\frac{31}{2.000} = \mathbf{15.500}$ |
| **4** | Movie D | **2** | $2^2 - 1 = \mathbf{3}$ | $\log_2(5) \approx \mathbf{2.322}$ | $\frac{3}{2.322} \approx \mathbf{1.292}$ |

Now sum all 4 discounted gains:
$$\text{DCG@4} = 7.000 + 0.000 + 15.500 + 1.292 = \mathbf{23.792}$$

### Step 2: Calculate the Ideal DCG (IDCG@4)

What is the **absolute best theoretical score** any model could achieve with these items?
Sort the items in **perfect descending order of relevance**:
$$\mathbf{rel}_{\text{ideal}} = [5, 3, 2, 0]$$

| Ideal Rank ($i$) | Ideal Item | Ideal Relevance ($\text{rel}_i$) | Numerator ($2^{\text{rel}} - 1$) | Denominator ($\log_2(i + 1)$) | Ideal Discounted Gain Added |
| :---: | :---: | :---: | :---: | :---: | :---: |
| **1** | Movie C | **5** | $2^5 - 1 = \mathbf{31}$ | $\log_2(2) = \mathbf{1.000}$ | $\frac{31}{1.000} = \mathbf{31.000}$ |
| **2** | Movie A | **3** | $2^3 - 1 = \mathbf{7}$ | $\log_2(3) \approx \mathbf{1.585}$ | $\frac{7}{1.585} \approx \mathbf{4.416}$ |
| **3** | Movie D | **2** | $2^2 - 1 = \mathbf{3}$ | $\log_2(4) = \mathbf{2.000}$ | $\frac{3}{2.000} = \mathbf{1.500}$ |
| **4** | Movie B | **0** | $2^0 - 1 = \mathbf{0}$ | $\log_2(5) \approx \mathbf{2.322}$ | $\frac{0}{2.322} = \mathbf{0.000}$ |

Now sum the ideal discounted gains:
$$\text{IDCG@4} = 31.000 + 4.416 + 1.500 + 0.000 = \mathbf{36.916}$$

### Step 3: Compute the Final NDCG@4

Divide what the model actually scored by the ideal maximum score:

$$\text{NDCG@4} = \frac{\text{DCG@4}}{\text{IDCG@4}} = \frac{23.792}{36.916} \approx \mathbf{0.6445} \quad (64.45\%)$$

---

### Why Did the Model Lose 35.55% of Its Score?
* In the **Ideal Ranking**, Movie C (the 5-star masterpiece) sits at Rank 1, earning a massive **$+31.00$ points**.
* In the **Model's Ranking**, Movie C was placed down at Rank 3. Its value was divided by $\log_2(4) = 2.0$, cutting its contribution down to only **$+15.50$ points**.
* That single ranking blunder cost the model **$15.50$ points**, dragging its total score down from $100\%$ to $64.45\%$. NDCG punished the model severely for making the user scroll to find the best item!

## 5. Mean NDCG@K Across All Users

Just like with MAP@K, in an offline evaluation dataset with $|U|$ users, we compute NDCG@K for each individual user $u$ and then take the **arithmetic mean**:

$$\text{Mean NDCG@K} = \frac{1}{|U|} \sum_{u=1}^{|U|} \text{NDCG@K}(u)$$

This produces a single, universally comparable benchmark metric for the entire recommendation engine.

## 6. Python Implementation

Here is a clean, production-ready Python implementation of DCG, IDCG, and NDCG@K using pure NumPy:

In [ ]:
import numpy as np

def compute_dcg_at_k(relevance_scores, k=10):
    """
    Computes Discounted Cumulative Gain at K (DCG@K).
    Uses the exponential gain formulation: (2^rel - 1) / log2(rank + 1).
    """
    rel = np.asfarray(relevance_scores)[:k]
    if rel.size == 0:
        return 0.0
    # Ranks from 1 to len(rel)
    ranks = np.arange(1, len(rel) + 1)
    discounts = np.log2(ranks + 1)
    gains = 2.0 ** rel - 1.0
    return np.sum(gains / discounts)

def compute_ndcg_at_k(relevance_scores, k=10):
    """
    Computes Normalized Discounted Cumulative Gain at K (NDCG@K).
    """
    dcg = compute_dcg_at_k(relevance_scores, k=k)
    
    # Ideal ranking is the relevance scores sorted in descending order
    ideal_relevance = sorted(relevance_scores, reverse=True)
    idcg = compute_dcg_at_k(ideal_relevance, k=k)
    
    if idcg == 0.0:
        return 0.0
    return dcg / idcg

# --- Verify with our exact movie example ---
model_ranking = [3, 0, 5, 2]

dcg_val = compute_dcg_at_k(model_ranking, k=4)
idcg_val = compute_dcg_at_k(sorted(model_ranking, reverse=True), k=4)
ndcg_val = compute_ndcg_at_k(model_ranking, k=4)

print(f"Calculated DCG@4  : {dcg_val:.4f} (Matches manual math: 23.792)")
print(f"Calculated IDCG@4 : {idcg_val:.4f} (Matches manual math: 36.916)")
print(f"Calculated NDCG@4 : {ndcg_val:.4f} (Matches manual math: 0.6445 or 64.45%)")

## 7. When and Where to Use NDCG@K

* **When to use:**
  * When you have **graded relevance feedback** (e.g., star ratings $1-5$, multi-level user engagement like click $= 1$, add-to-cart $= 3$, purchase $= 5$).
  * When position order is critical, and you want a smooth, diminishing-returns penalty for lower positions.
  * When you need a metric strictly normalized between $0.0$ and $1.0$ to compare across users who have different numbers of relevant items.
* **Where to use in industry:**
  * **Streaming Platforms (Netflix, Spotify, YouTube):** Ranking video rows and personalized playlists.
  * **Search Engines (Google, Bing):** The standard benchmark metric for Learning-to-Rank (LTR).
  * **E-Commerce Marketplaces (Amazon, eBay):** Product search ranking and recommendation widgets.
  * **Learning-to-Rank ML Models:** Direct optimization target for **LambdaMART**, **LightGBM Ranker**, and **XGBoost Ranker**.

## 8. Pros and Cons of NDCG@K

### Pros:
1. **Handles Nuanced Graded Relevance:** Naturally accommodates multiple degrees of relevance rather than simplistic binary flags.
2. **Smooth Logarithmic Decay:** The $\log_2$ discount realistically simulates user attention drop-off down a screen.
3. **Perfect Normalization:** Always produces a score between $0.0$ and $1.0$, preventing active power users with hundreds of ratings from biasing the global evaluation score.
4. **Direct Alignment with Learning-to-Rank:** Modern ranking algorithms can directly optimize smooth approximations of NDCG (e.g., LambdaRank / LambdaMART).

### Cons:
1. **Requires Graded Labels:** Demands granular relevance data (e.g., ratings or engagement tiers), which can be costly or noisy to acquire.
2. **Missing/Unseen Items Issue:** If an item in the top $K$ has not yet been rated by the user in the test set, it is assumed to have relevance $= 0$, which can falsely penalize the model for recommending undiscovered gems.
3. **Heuristic Formulation:** The logarithmic discount base ($\log_2$) is an empirical human-behavior heuristic rather than a derived natural law.

## 9. Master Comparison: Precision@K vs. MRR@K vs. MAP@K vs. NDCG@K

| Evaluation Metric | Evaluates Multiple Hits? | Order / Rank Aware? | Relevance Scale Supported | Core Question Answered |
| :--- | :---: | :---: | :---: | :--- |
| **Precision@K** | ✅ Yes | ❌ No | Binary ($0/1$) | *"What percentage of the top $K$ items are relevant?"* |
| **MRR@K** | ❌ No (Stops at 1st) | ✅ Yes | Binary ($0/1$) | *"How high up on the screen was the **first** hit?"* |
| **MAP@K** | ✅ **Yes** | ✅ **Yes** | Binary ($0/1$) | *"What was the precision at **each** relevant hit?"* |
| **NDCG@K** | ✅ **Yes** | ✅ **Yes** | **Graded ($0$ to $5$ stars, or click/cart/buy)** | *"Are the **absolute best items** at the very top, accounting for nuance?"* |

## 10. Frequently Asked Question: "What happens if a user has zero relevant items in the test set?"

If a user has no relevant interactions in the test set, their ideal ranking consists entirely of zeros:
$$\mathbf{rel}_{\text{ideal}} = [0, 0, \dots, 0] \implies \text{IDCG@K} = 0.0$$

Dividing $\frac{\text{DCG@K}}{\text{IDCG@K}} = \frac{0}{0}$ is mathematically undefined. 

**Standard Industry Protocol:**
In production libraries (such as Scikit-Learn's `ndcg_score`), when $\text{IDCG} = 0$, the NDCG score for that user is explicitly defined as **$0.0$** (or that user is omitted from the evaluation set if they had no test interactions).